# Day 20 — Solution: The Mixed Problem Set

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "qrc").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
from scipy import stats
DATA_SOURCE = os.environ.get("QRC_DATA", "real")
from qrc.data import get_prices
from qrc.synth import synthetic_prices

if DATA_SOURCE == "real":
    px = get_prices(["SPY", "TLT"], start="1990-01-01")
else:
    px = synthetic_prices(n_days=3000, n_assets=2, seed=83, corr=0.2)
    px.columns = ["SPY", "TLT"]
    px["TLT"] = -px["TLT"]
r = px.pct_change().dropna()
rng = np.random.default_rng(77)

## P1 — the paired audit

In [ ]:
sA, sB, rho, n = 0.014, 0.011, 0.72, 800
muA, muB = 0.00031, 0.00019
sd_d = np.sqrt(sA**2 + sB**2 - 2*rho*sA*sB)
se_paired = sd_d/np.sqrt(n)
t = (muA - muB)/se_paired
se_welch = np.sqrt(sA**2/n + sB**2/n)
t_w = (muA - muB)/se_welch
print(f"sd(diff) = {sd_d*1e4:.2f}bp | paired SE {se_paired*1e4:.3f}bp | t {t:.2f}")
print(f"Welch SE {se_welch*1e4:.3f}bp | t {t_w:.2f} | ratio {se_welch/se_paired:.2f}")

**Expected reasoning.** First line is the design: "matched dates,
shared market → paired." sd(diff) = √(sA²+sB²−2ρsAsB) = 0.976%;
SE = 0.976%/√800 = 3.45bp; t = 1.2bp/3.45bp = **0.35**. Welch:
SE = 6.29bp, t = 0.19, ratio 1.82 — the pairing dividend at
ρ = 0.72 (√3.33). Verdict: nothing here; and the honest sentence
is the paired one (the Welch number is smaller *in significance*,
not bigger — pairing helps a real edge, it doesn't manufacture one).

## P2 — the bootstrap that must match

In [ ]:
x = r.iloc[:, 0].values
n = len(x)
idx = rng.integers(0, n, size=(5000, n))
se_boot = np.array([x[i].mean() for i in idx]).std()
se_text = x.std(ddof=1)/np.sqrt(n)
ac = np.array([pd.Series(x).autocorr(k) for k in range(1, 21)])
neff = n/(1 + 2*ac.sum())
se_clust = x.std(ddof=1)/np.sqrt(neff)
print(f"naive boot {se_boot*1e4:.3f}bp | textbook {se_text*1e4:.3f}bp | "
      f"clustered {se_clust*1e4:.3f}bp | n_eff {neff:,.0f} ({neff/n:.2f}n)")

**Expected reasoning.** (a) ≈ (b) within Monte-Carlo noise (the
bootstrap must reproduce the known SE — the unit test); (c) larger
by ≈ √(n/n_eff) ≈ 1.2–1.7. The spread (a) vs (c) is exactly the
clustering the single-day resampler destroyed — the module 04.9
result on real data.

## P3 — the permutation on the right null

In [ ]:
x, y = r.iloc[:, 0].values, r.iloc[:, 1].values
rho_obs = np.corrcoef(x, y)[0, 1]
B = 2000
rhos = np.array([np.corrcoef(x, rng.permutation(y))[0, 1] for _ in range(B)])
p_perm = (np.sum(np.abs(rhos) >= abs(rho_obs)) + 1)/(B + 1)
t_f = rho_obs*np.sqrt((len(x) - 2)/(1 - rho_obs**2))
p_f = 2*stats.t.sf(abs(t_f), len(x) - 2)
print(f"rho {rho_obs:.3f} | perm p {p_perm:.4f} | Fisher p {p_f:.4f}")

**Expected reasoning.** Both p's tiny (ρ ≈ −0.2 to −0.35 on real
SPY/TLT) and close in size. Close *because* the returns are
near-iid (ACF ≈ 0): shuffling y destroys almost no structure the
correlation test depends on. The permutation version is **wrong** on
strongly autocorrelated data (prices, vol, any clustered series):
the shuffled null has no clustering, the statistic looks too
extreme, the p too small. Check the ACF before you trust the
permutation p.

## P4 — the screen

In [ ]:
m, n = 80, 1000
pvals, ts = [], []
for _ in range(m):
    ret = rng.normal(0, 0.04, n)
    lab = rng.choice([0, 1], n)
    hi, lo = ret[lab == 1], ret[lab == 0]
    t = (hi.mean() - lo.mean())/np.sqrt(hi.var()/len(hi) + lo.var()/len(lo))
    ts.append(t); pvals.append(2*stats.t.sf(abs(t), len(hi)+len(lo)-2))
ts, p = np.array(ts), np.sort(np.array(pvals))
ks = np.arange(1, m+1)
k = np.max(ks[p <= ks*0.05/m]) if (p <= ks*0.05/m).any() else 0
print(f"|t|>2 by noise: {(np.abs(ts) > 2).sum()} (theory {m*0.0455:.1f}) | BH rejects {k}")

**Expected reasoning.** ≈ 3–4 at |t| > 2 (theory 3.6). BH rejects
0–2 (its q = 0.05 target means the declared set is *mostly false*
in a pure-noise screen: every survivor is, by construction, false).
The honest sentence: "BH at q = 0.05 controls the *expected*
fraction false across the declared set — in a screen with nothing
true, the survivors are the expected false discoveries, and the
procedure's job was to count them, not to certify them."

## P5 — the overlapping audit

In [ ]:
H, N = 21, 1800
# construct the overlapping series from 21-day blocks of iid daily
d = rng.normal(0, 0.0012, N + H)                     # daily, ~1.2bp/day scaled
R = pd.Series(d).rolling(H).sum().dropna().values[:N]
# rescale to the stated design (mean 6bp, sd 28bp per 21-day obs)
R = (R - R.mean())/R.std() * 0.0028 + 0.0006

a = [np.corrcoef(R[:-k], R[k:])[0, 1] for k in range(1, 21)]
mult = np.sqrt(1 + 2*sum(a))
t_naive = R.mean()/(R.std()/np.sqrt(len(R)))
print(f"naive t {t_naive:.2f} | factor {mult:.2f} (sqrt(H) {np.sqrt(H):.2f}) | corrected {t_naive/mult:.2f}")
non = R[::H]
t_non = non.mean()/(non.std()/np.sqrt(len(non)))
print(f"non-overlapping: N = {len(non)}, t = {t_non:.2f}")

**Expected reasoning.** Naive t ≈ 9; factor ≈ √21 ≈ 4.58 (the
rescaled series has the triangle ACF by construction); corrected
t ≈ 1.9–2.0. Non-overlapping: N = 85, same per-observation mean/SD,
t ≈ 1.9–2.0 — **the two designs agree** (within sampling noise):
overlap never creates information, it only mislabels the
accounting. A 9-t that is really a 2-t: the checkpoint's standard
warning, quantified.

## P6 — the full audit (exemplar)

**Grill.** (1) Effect: 1.1%/yr, SE = 110/2.7 ≈ 41bp/yr, CI
[29, 191]bp — positive, band 73% of the effect. (2) Power:
detectable ≈ 2.8 × 41 ≈ 115bp/yr — the reported 110bp sits *just
below* its own 80%-power line: a 90bp effect would have been missed
~50% of the time. (3) Family: m = 9 → Bonferroni threshold ≈ 2.77:
**2.7 fails by a hair**; expected false t's ≥ 2.7 among 9 noise
claims ≈ 9 × 0.007 ≈ 0.06. (4) Design: quarterly, non-overlapping —
clean; no overlap flag; the one residual is factor clustering in
quintile portfolios (ask, minor).

**Gate.** 4 events/yr × 12bp = 48bp/yr cost. Net = 110 − 48 = 62bp/
yr; net Sharpe = 62/1800 ≈ **0.034**. Death cost = 110/4 =
27.5bp/event — 2.3× the assumed 12bp: a real but *thin* margin.

**Disposition.** "The grill: a positive-CI candidate that fails its
own family by 0.05 in t and sits below its own power line — a
hypothesis with a clean design, not a discovery. The gate: alive at
12bp (net 62bp/yr, Sharpe 0.03) with 2.3× margin to death —
economically marginal. Funded only under: (1) a disjoint-sample
replication at corrected t ≥ 3.0 with the family pre-declared, and
(2) the replication's effect clearing the 115bp detection line.
Otherwise: watchlist, one replication budget, no position."

The structure to copy: *design → honest number → family → cost →
conditional verdict.* That one paragraph is the checkpoint (day 21).